# Train and segment a tiled area

This notebook uses the new top-level API for the common workflow:

- train one bundled model from the clipped labeled training package
- build tiled inference inputs from the larger clipped rasters
- segment those prepared tiles with the bundled model

The tiled path is the important part here: the full inference scene is segmented tile by tile, while StarDist can still use smaller internal refinement windows inside each prepared tile.


## Inputs

Training uses the clipped labeled training package:

- `data/training_data_strong/img.tif`
- `data/training_data_strong/chm.tif`
- `data/training_data_strong/den.tif`
- `data/training_data_strong/crowns.gpkg`

Inference uses the larger clipped rasters, which are then prepared into tiled inputs.


In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "samstars").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from samstars import train, segment
from samstars.data import build_tiles


/Users/iosefa/miniforge3/envs/samstars/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Set paths


In [2]:
data_dir = repo_root / "data"
training_data_dir = data_dir / "training_data_strong"

training_img = training_data_dir / "img.tif"
training_chm = training_data_dir / "chm.tif"
training_den = training_data_dir / "den.tif"
training_crowns = training_data_dir / "crowns.gpkg"

img = data_dir / "img_clipped.tif"
chm = data_dir / "chm_clipped.tif"
den = data_dir / "den_clipped.tif"

sam_base_ckpt = repo_root / "sam/sam_vit_l_0b3195.pth"
work_dir = data_dir / "train_and_segment_tiled_area_run"
model_dir = work_dir / "model"
tiles_dir = work_dir / "tiles"
inference_run_dir = work_dir / "segmentation_run"

print("repo_root:", repo_root)
print("training_data_dir:", training_data_dir)
print("model_dir:", model_dir)
print("tiles_dir:", tiles_dir)
print("inference_run_dir:", inference_run_dir)


repo_root: /Users/iosefa/repos/sam
training_data_dir: /Users/iosefa/repos/sam/data/training_data_strong
model_dir: /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/model
tiles_dir: /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/tiles
inference_run_dir: /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/segmentation_run


## Set tutorial parameters


In [3]:
rgb_bands = (1, 2, 3)

device = "mps"
storage = "persistent"

tile_size_m = 500.0
tile_overlap_m = 20.0

chip_size = 512
chip_stride = 384

sam_epochs = 3
stardist_epochs = 10
val_fraction = 0.2

crop_smooth_sigma = 2.0
smooth_sigma = 4.0
post_smooth_sigma = 0.0

thin_seeds = False
thin_dist = 0.8
seed_d_min = 4.0
seed_min_dist_px = 4
seed_gauss_sigma = 1
seed_min_samples = 1
seed_merge_radius = 3.0
seed_min_canopy_height = 2.0

stardist_input_key = "index"
stardist_use_aux = True
stardist_prob_thr = 0.3
stardist_nms_thr = 0.4
stardist_window_size_m = 150.0
stardist_window_overlap_m = 30.0
stardist_dedup_iou_thr = 0.6


## Train bundled model

This keeps the persistent training state under `model_dir/_training/` and writes the flat bundle files at the top level of `model_dir`.


In [4]:
model = train(
    img=training_img,
    chm=training_chm,
    den=training_den,
    crowns=training_crowns,
    sam_base_ckpt=sam_base_ckpt,
    out=model_dir,
    storage=storage,
    rgb_bands=rgb_bands,
    device=device,
    chip_size=chip_size,
    chip_stride=chip_stride,
    sam_epochs=sam_epochs,
    stardist_epochs=stardist_epochs,
    val_fraction=val_fraction,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
    thin_seeds=thin_seeds,
    thin_dist=thin_dist,
    seed_d_min=seed_d_min,
    seed_min_dist_px=seed_min_dist_px,
    seed_gauss_sigma=seed_gauss_sigma,
    seed_min_samples=seed_min_samples,
    seed_merge_radius=seed_merge_radius,
    seed_min_canopy_height=seed_min_canopy_height,
    stardist_input_key=stardist_input_key,
    stardist_use_aux=stardist_use_aux,
    stardist_window_size_m=stardist_window_size_m,
    stardist_window_overlap_m=stardist_window_overlap_m,
    stardist_dedup_iou_thr=stardist_dedup_iou_thr,
    fast=False,
)


bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
function_base.py (4824): Warning: 'partition' will ignore the 'mask' of the MaskedArray.
cost_surface.py (137): No SLIC provided – cost built from 3 terms only.
_base.py (248): Precomputed sparse input was not sorted by row values. Use the function sklearn.neighbors.sort_graph_by_row_values to sort the input by row values, with warn_when_not_sorted=False to remove this warning.


✓ Single-tile inputs written to /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/model/_training/prep/scene
tile_000_000: wrote 147 training chips
Built 147 training chips in /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/model/_training/training_chips
training_chips: epochs=3 (iters/epoch=147)
training_chips epoch 1/3 loss=0.4402
training_chips epoch 2/3 loss=0.2706
training_chips epoch 3/3 loss=0.2508
✓ weights → /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/model/decoder.pth
seeds in chip: 486
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 7.88346887e-01
 9.99795794e-01]
seed prob pcts: [1.58057983e-10 3.05771266e-07 4.13909629e-06 5.78221639e-01
 8.52778614e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.12115631e-06 7.11734002e-01
 9.54231441e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.22819074e-06 8.33636267e-01
 9.86272156e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 4.78818156e-06 7.48017566e-01
 

proposals.py (105): invalid value encountered in divide
2026-04-20 18:33:55.669381: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Max
2026-04-20 18:33:55.669411: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-04-20 18:33:55.669414: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 48.00 GB
2026-04-20 18:33:55.669424: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-20 18:33:55.669432: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


prob_s pcts: [0.00000000e+00 9.56727745e-05 3.51974535e-01 6.85513999e-01
 8.63837779e-01]
✓ wrote proposal outputs (1 records)
✓ wrote /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/model/_training/stardist_training_records/train_records.json (1 records)
✓ wrote /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/model/_training/stardist_training_records/val_records.json (0 records)
epochs=10 (iters/epoch=1)
Using default values: prob_thresh=0.5, nms_thresh=0.4.
no validation records found; reusing training data as validation_data
Epoch 1/10


2026-04-20 18:33:56.840814: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


100/100 ━━━━━━━━━━━━━━━━━━━━ 27s 240ms/step - dist_dist_iou_metric: 0.2457 - dist_loss: 6.4691 - dist_relevant_mae: 6.4685 - dist_relevant_mse: 69.0079 - loss: 1.7409 - prob_kld: 0.2957 - prob_loss: 0.4471 - val_dist_dist_iou_metric: 0.4121 - val_dist_loss: 4.8117 - val_dist_relevant_mae: 4.8107 - val_dist_relevant_mse: 35.6627 - val_loss: 1.3367 - val_prob_kld: 0.2162 - val_prob_loss: 0.3744 - learning_rate: 3.0000e-04
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 23s 233ms/step - dist_dist_iou_metric: 0.3846 - dist_loss: 4.8449 - dist_relevant_mae: 4.8440 - dist_relevant_mse: 37.8196 - loss: 1.2841 - prob_kld: 0.1628 - prob_loss: 0.3152 - val_dist_dist_iou_metric: 0.4087 - val_dist_loss: 4.6670 - val_dist_relevant_mae: 4.6661 - val_dist_relevant_mse: 34.5447 - val_loss: 1.1723 - val_prob_kld: 0.0807 - val_prob_loss: 0.2389 - learning_rate: 3.0000e-04
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 23s 233ms/step - dist_dist_iou_metric: 0.4042 - dist_loss: 4.6166 - dist_relevant_mae: 4.6159 - dist_

## Inspect bundle and persistent training outputs


In [5]:
sorted(p.name for p in model.root.iterdir())


['_training', 'config.json', 'decoder.pth', 'manifest.json', 'weights.h5']

In [6]:
training_proposals_dir = model.root / "_training" / "training_run" / "proposals" / "rasters"
sorted(p.name for p in training_proposals_dir.glob("*.tif"))[:40]


['tile_000_000_0_chm.tif',
 'tile_000_000_0_cost.tif',
 'tile_000_000_0_logit.tif',
 'tile_000_000_0_sam_index.tif',
 'tile_000_000_0_sam_prob.tif']

## Build inference tiles


In [7]:
build_tiles(
    img=img,
    chm=chm,
    den=den,
    rgb_bands=rgb_bands,
    out=tiles_dir,
    tile=tile_size_m,
    overlap=tile_overlap_m,
    thin_seeds=thin_seeds,
    thin_dist=thin_dist,
    seed_d_min=seed_d_min,
    seed_min_dist_px=seed_min_dist_px,
    seed_gauss_sigma=seed_gauss_sigma,
    seed_min_samples=seed_min_samples,
    seed_merge_radius=seed_merge_radius,
    seed_min_canopy_height=seed_min_canopy_height,
    skip_existing=True,
)


Tiling:   0%|                                                  | 0/4 [00:00<?, ?tile/s]function_base.py (4824): Warning: 'partition' will ignore the 'mask' of the MaskedArray.
cost_surface.py (137): No SLIC provided – cost built from 3 terms only.
_base.py (248): Precomputed sparse input was not sorted by row values. Use the function sklearn.neighbors.sort_graph_by_row_values to sort the input by row values, with warn_when_not_sorted=False to remove this warning.
Tiling:  25%|██████████▌                               | 1/4 [00:03<00:09,  3.32s/tile]function_base.py (4824): Warning: 'partition' will ignore the 'mask' of the MaskedArray.
cost_surface.py (137): No SLIC provided – cost built from 3 terms only.
_base.py (248): Precomputed sparse input was not sorted by row values. Use the function sklearn.neighbors.sort_graph_by_row_values to sort the input by row values, with warn_when_not_sorted=False to remove this warning.
Tiling:  50%|█████████████████████                     | 2/4 [00

✓ Tiles written to /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/tiles


## Segment tiled inference area


In [ ]:
result = segment(
    model=model,
    out=inference_run_dir,
    storage=storage,
    tiles_dir=tiles_dir,
    device=device,
    stardist_input_key=stardist_input_key,
    stardist_use_aux=stardist_use_aux,
    stardist_prob_thr=stardist_prob_thr,
    stardist_nms_thr=stardist_nms_thr,
    stardist_window_size_m=stardist_window_size_m,
    stardist_window_overlap_m=stardist_window_overlap_m,
    stardist_dedup_iou_thr=stardist_dedup_iou_thr,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
    fast=False,
)


## Final output


In [ ]:
result.crowns
